# Iceberg → tokenized Parquet change batches for Databricks

The nightly DAG task `export_databricks` (`cde/jobs/export_parquet.py`) hands the silver tables to
Databricks as an **incremental, tokenized Parquet feed**:

- Each run compares the current Iceberg snapshot with the one exported last, and writes only the
  changed rows: `_op` = I (new key), U (changed row), D (key gone).
- `cif`, `account_id` and `event_id` become deterministic AES-256-GCM tokens. Pincode is cut to 3
  digits and dates to the month. `config/export.yaml` lists every column, deny by default.
- Databricks applies the batches as SCD type 2 tables and decrypts with its built-in
  `aes_decrypt(..., 'GCM')`, for one account group only (`databricks/README.md`).

**This notebook is a safe demo.** It uses the job's own functions, but with a **throwaway key** made
in this session and a **scratch folder** (`databricks_export_notebook/`). It never touches the
production feed, its key or the source table's `databricks_export` tag. The production run is the CDE
job (last cell).

| | |
|---|---|
| Source | `rsingh_spend_anl_silver.customer` (Iceberg v2, ~195k rows); any table in `config/export.yaml` works |
| Scratch target | `s3a://federal-buk-574bcea0/data/rsingh_spend_anl/databricks_export_notebook/` |
| Production feed | `s3a://federal-buk-574bcea0/data/rsingh_spend_anl/databricks_export/` (CDE job only) |

**Session:** New Session › JupyterLab, Python 3.11 › **Enable Spark: Spark 3.5.4 - CDE 1.25.2** ›
2 vCPU / 8 GB.

In [ ]:
import base64
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "cde" / "jobs"))
import export_parquet as ex

DB_PREFIX = "rsingh_spend_anl"
TABLE = "silver.customer"                # or "silver.account", "silver.spend_event" (30M rows: slower)
CFG = ex.load_config(ROOT / "config" / "export.yaml")
TCFG = CFG["tables"][TABLE]
SCRATCH = CFG["target"].rstrip("/") + "_notebook"
assert SCRATCH.endswith("databricks_export_notebook") and SCRATCH != CFG["target"]
BUCKET = "/".join(SCRATCH.split("/")[:3])

# A throwaway key for this session only: not the feed's key, never written anywhere.
KEYS = {"version": "notebook-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S"),
        "enc": os.urandom(32), "iv": os.urandom(32)}
print("source:", f"{DB_PREFIX}_{TABLE}", "| scratch target:", SCRATCH, "| demo key", KEYS["version"])

## 1. Spark with Iceberg

The Spark add-on ships the Iceberg runtime in `/opt/spark/optional-lib`. `SparkSessionCatalog` puts
Iceberg in front of the Hive metastore, so the tables resolve by the same names the CDE jobs use.
`spark.kerberos.access.hadoopFileSystems` gets the session an IDBroker token for the bucket.

In [ ]:
import glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

jars = ",".join(sorted(glob.glob("/opt/spark/optional-lib/iceberg-*.jar")))
builder = (SparkSession.builder.appName("spend-export-parquet-notebook")
           .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
           .config("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkSessionCatalog")
           .config("spark.sql.catalog.spark_catalog.type", "hive")
           .config("spark.hadoop.iceberg.engine.hive.enabled", "true")
           .config("spark.kerberos.access.hadoopFileSystems", BUCKET)
           .config("spark.sql.session.timeZone", "UTC")
           .config("spark.executor.instances", "2").config("spark.executor.cores", "2")
           .config("spark.executor.memory", "6g"))
if jars:
    builder = builder.config("spark.jars", jars)
spark = builder.getOrCreate()
print("Spark", spark.version, "| Iceberg jars:", jars or "already on the classpath")

## 2. The source: snapshots

Silver is rebuilt every night, so each night adds a snapshot. The export diffs two of them: the current
one, and the one the last export read (kept from expiring by the `databricks_export` tag).

In [ ]:
layer, table = TABLE.split(".")
FQ = f"{DB_PREFIX}_{layer}.{table}"
SID = ex.current_snapshot(spark, FQ)
print(f"{FQ}: current snapshot {SID}")
spark.sql(f"SELECT made_current_at, snapshot_id, is_current_ancestor FROM {FQ}.history "
          "ORDER BY made_current_at DESC LIMIT 5").show(truncate=False)
spark.sql(f"SELECT name, type, snapshot_id, max_reference_age_in_ms / 86400000 AS retain_days FROM {FQ}.refs").show(truncate=False)

## 3. What leaves Cloudera: the column rules

Every source column must have a rule in `config/export.yaml`; an unlisted column fails the export.
The side-by-side below exists **only in this session** (raw values are never written).

In [ ]:
cur = ex.at_snapshot(spark, FQ, SID)
ex.check_columns(cur.columns, TCFG, FQ)
print({c: r for c, r in TCFG["columns"].items() if r != "keep"}, "| everything else: keep")

enc = ex.encrypt_udf(KEYS)
sample = cur.limit(3)
side = sample.select(*[c for c in sample.columns if TCFG["columns"][c] != "keep"])
tok = ex.apply_rules(side, TCFG, enc)
side.toPandas().join(tok.toPandas(), lsuffix="  (raw)", rsuffix="  (exported)")

**Deterministic:** the same `cif` always gives the same token, so `cif` joins across `customer`,
`account` and `spend_event` without decrypting. The IV comes from a second key (`iv_key`) that never
leaves Cloudera, so a token can't be made for a guessed value on the Databricks side.

In [ ]:
v = cur.select("cif").first()["cif"]
t1, t2 = ex.tokenize_value(v, KEYS["enc"], KEYS["iv"]), ex.tokenize_value(v, KEYS["enc"], KEYS["iv"])
print("same value, same token:", t1 == t2, "|", t1)
print("decrypts back:", ex.detokenize_value(t1, KEYS["enc"]) == v)

## 4. The change set between two real nightly snapshots

The same diff the job runs: join on the primary key, compare a hash of the other columns. Counts only;
nothing is written. The nightly `createOrReplace` starts a new snapshot lineage
(`is_current_ancestor` is false for yesterday's snapshot), but time travel by snapshot ID still reads it.

In [ ]:
prev_sids = [r["snapshot_id"] for r in spark.sql(
    f"SELECT snapshot_id FROM {FQ}.history WHERE snapshot_id != {SID} "
    "ORDER BY made_current_at DESC LIMIT 1").collect()]
if prev_sids and ex.readable(spark, FQ, prev_sids[0]):
    prev = ex.at_snapshot(spark, FQ, prev_sids[0])
    changes = ex.change_set(cur, prev, ex.primary_key(TCFG))
    print(f"snapshot {prev_sids[0]} -> {SID}:")
    changes.groupBy("_op").count().orderBy("_op").show()
else:
    print("no earlier readable snapshot: the first export is a full batch")

## 5. Export batch 1 (full) to the scratch folder

The scratch folder for this table is cleared first, so this is always batch 1. The job writes the
Parquet, reads it back and checks the count, then writes the manifest **last** (the commit marker).
`tag=None`: the source table's `databricks_export` tag stays where the production feed put it.

In [ ]:
TPATH = ex.table_path(SCRATCH, layer, table)
p, fs = ex._fs(spark, TPATH)
if fs.exists(p):
    fs.delete(p, True)

m = ex.export_table(spark, DB_PREFIX, TABLE, TCFG, SCRATCH, KEYS, tag=None)
print(json.dumps({k: m[k] for k in ("batch", "batch_type", "reason", "from_snapshot_id", "to_snapshot_id",
                                    "source_row_count", "counts", "key_version", "tokenized_columns",
                                    "data_path", "files", "bytes")}, indent=1))

## 6. Run it again: nothing changed, nothing written

Same snapshot as the last manifest, so the job writes no batch. Tonight's rebuild gives a new
snapshot, and the next run writes only the I / U / D rows of section 4.

In [ ]:
again = ex.export_table(spark, DB_PREFIX, TABLE, TCFG, SCRATCH, KEYS, tag=None)
print(again["batch_type"], "| snapshot", again["to_snapshot_id"])

## 7. Read it the way Databricks will

Plain Parquet, no Iceberg. Decrypt with the built-in `aes_decrypt(unbase64(tok), key, 'GCM')`, the
same call Databricks' `detok()` makes (`databricks/03_detokenize.sql`), and join back to the source.
Every row should match. The demo key appears in this session's Spark plan; that's acceptable for a
throwaway key, but not for the feed's key, which is why Databricks reads it with `secret()`.

In [ ]:
pq = spark.read.parquet(m["data_path"])
pq.printSchema()
key_hex = KEYS["enc"].hex()
dec = pq.select(F.expr(f"CAST(aes_decrypt(unbase64(cif), unhex('{key_hex}'), 'GCM') AS STRING)").alias("cif"))
matched = dec.join(cur.select("cif"), "cif").count()
print(f"{pq.count():,} exported rows, {matched:,} decrypt to a cif in the source")
pq.drop("_exported_at").limit(5).toPandas()

In [ ]:
root = spark._jvm.org.apache.hadoop.fs.Path(TPATH)
base = str(fs.makeQualified(root)) + "/"
it = fs.listFiles(root, True)
rows = []
while it.hasNext():
    s = it.next()
    rows.append((str(s.getPath()).replace(base, ""), round(s.getLen() / 1e6, 2)))
for name, mb in sorted(rows)[:12]:
    print(f"  {mb:>8} MB  {name}")
print(ex.read_text(spark, f"{TPATH}/_manifests/{ex.batch_name(m['batch'])}.json")[:1200])

## 8. On the Databricks side

`databricks/` in this repo, explained in `databricks/README.md`:

| File | What |
|---|---|
| `01_setup.sql` | external location on the feed, schemas, the `spend_tokens` secret scope (enc_key only) |
| `02_ingest_scd2.sql` | Lakeflow pipeline: Auto Loader on `data/` → `AUTO CDC ... STORED AS SCD TYPE 2` |
| `03_detokenize.sql` | `detok()` for the `spend_pii_readers` group, views of current rows, column-mask option |
| `ingest_manifest_driven.py` | the strict alternative: only committed batches, resync handled in place |

```sql
CREATE FLOW customer_scd2 AS AUTO CDC INTO spend_silver.customer
FROM STREAM(spend_bronze.customer_changes)
KEYS (cif)
APPLY AS DELETE WHEN _op = 'D'
SEQUENCE BY _batch
COLUMNS * EXCEPT (_op, _source_snapshot_id, _exported_at, _file)
STORED AS SCD TYPE 2;
```

## The production feed

The nightly DAG runs it after `dq_gold`, beside the ML jobs. By hand:

```bash
cde job run --name rsingh-spend-anl-export-parquet --arg=--dry-run       # counts only
cde job run --name rsingh-spend-anl-export-parquet                       # all three tables
```

The job reads the real key from the CDE workload credential `rsingh-spend-anl-token-key`.

In [ ]:
fs.delete(p, True)          # remove the scratch export
spark.stop()